# Feature Lab

Experimental analytics on top of the `cricbot` modules: **momentum**,
an **impact scorecard**, and an outcome-derived **pitch profile**.

Logic lives in `impact_lab.py` so it can be tuned in one place; this
notebook exercises it and holds the open questions.

Requires network access to ESPNCricinfo.


In [ ]:
from cricdata import CricinfoClient
from impact_lab import (
    turning_points, momentum_series, match_par,
    batting_impact, bowling_impact, impact_leaderboard,
    pitch_profile, bowler_kind,
)
from scorecard_charts import sparkline
from scorecard_helpers import innings_team_name
from cricbot_views import match_slugs

ci = CricinfoClient()

SERIES = 'ipl-2025-1449924'

results = ci.series_matches(SERIES)
matches = results['content']['matches']

m = matches[0]
series_slug, match_slug = match_slugs(m)

info = ci.match_info(series_slug, match_slug)
scorecard = ci.match_scorecard(series_slug, match_slug)
innings = scorecard['content']['innings']

print(m['title'], '|', m['statusText'])


## 1. Turning points

The overs with the largest swing in win probability.

`predictions.winProbability` is **the batting side's**, so a swing is
read from the perspective of whoever is batting in that innings.


In [ ]:
for e in turning_points(innings, top_n=5):
    print(f"{e['team']} over {e['over']:>2}: {e['swing']:+6.1f}pp "
          f"-> {e['win_prob']:5.1f}%  ({e['runs']} runs, {e['wickets']}w)")


## 2. Momentum

EWMA of each over's deviation from the innings' own par run rate,
with wickets weighted heavily. Smoothing matters — without it this is
just the manhattan chart again.

Tune `alpha` (persistence) and `w_wkt` (wicket severity).


In [ ]:
for inn in innings:
    mom = momentum_series(inn, alpha=0.3, w_wkt=8.0)
    print(f"{innings_team_name(inn):<5} {sparkline(mom)}")
    print(f"      {min(mom):+.1f} .. {max(mom):+.1f}")

# Sensitivity to alpha
for a in (0.15, 0.3, 0.5):
    print(f"alpha={a}: {sparkline(momentum_series(innings[0], alpha=a))}")


## 3. Impact scorecard

Two design decisions:

1. **Everything in runs above par**, so batters and bowlers rank on one
   scale.
2. **Par is derived from the match itself** (`total runs / total balls`),
   so the pitch normalizes out with no venue model.

```
BatImpact  = (runs - balls x par_rpb) + entry_bonus - wicket_runs (if out)
BowlImpact = (balls x par_rpb - conceded) + wickets x wicket_runs
```

`wicket_runs` is charged to batters and credited to bowlers, so a match
sums to ~zero. The residual is extras and run-outs.


In [ ]:
par = match_par(innings)
print(f"{par['runs']} runs off {par['balls']} balls")
print(f"par SR {par['par_sr']:.1f} | par RPB {par['par_rpb']:.3f}")

print()
for r in impact_leaderboard(innings, par, wicket_runs=12.0)[:10]:
    if r['role'] == 'bat':
        detail = f"{r['runs']}({r['balls']})" + ('' if r['out'] else '*')
    else:
        detail = f"{r['wickets']}/{r['conceded']} in {r['overs']}"
    print(f"{r['impact']:+7.1f}  {r['short']:<14}{r['role']:<5}{detail}")


### Open question: calibrating `wicket_runs`

This is the one free parameter and there is no obviously right value.

- `W = 0` makes batting and bowling leaderboards comparable in range,
  but ignores that taking wickets wins matches.
- `W = 12` reflects a wicket's run-equivalent cost in T20, but bowlers
  then dominate the top of the table — 11 wickets are credited across
  ~6 bowlers, while the cost is spread across 11 batters.

The zero-sum residual should stay **constant** as W varies. If it
doesn't, the charge and credit have drifted out of sync.


In [ ]:
print(' W    sum(bat)  sum(bowl)  residual   top-bat  top-bowl')
for W in (0, 6, 12, 18, 24):
    bat = batting_impact(innings, par, wicket_runs=W)
    bowl = bowling_impact(innings, par, wicket_runs=W)
    sb = sum(r['impact'] for r in bat)
    sw = sum(r['impact'] for r in bowl)
    print(f'{W:>2}  {sb:+9.1f} {sw:+9.1f} {sb+sw:+9.1f}   '
          f'{bat[0]["impact"]:+7.1f}  {bowl[0]["impact"]:+7.1f}')


## 4. Pitch profile

Outcome-derived only — no ball-tracking is available, so this reads the
pitch from what happened, not from trajectories.

Pace/spin split comes from `bowlingStyles` codes (`rfm`, `sla`, `lbg`...).

**A single match is not a pitch rating.** Aggregate 10+ matches per venue
and z-score against the season baseline before drawing conclusions.


In [ ]:
pp = pitch_profile(innings)

for kind in ('pace', 'spin'):
    d = pp[kind]
    print(f"{kind:<5} {d['balls']:>4} balls  econ {d['econ']}  "
          f"{d['wkts']}w  {d['dots']} dots")

print(f"spin - pace econ : {pp['spin_minus_pace_econ']:+}")
print(f"innings RPO      : {pp['innings_rpo']}")
print(f"deterioration    : {pp['deterioration']:+}")
print(f"dismissals       : {pp['dismissals']}")

# Who bowled what
for b in innings[0]['inningBowlers']:
    p = b['player']
    print(f"  {p['name']:<20} {bowler_kind(p):<8} {p.get('longBowlingStyles')}")


### Scaling beyond one match

An earlier version of this notebook proposed backfilling a season of
scorecards to characterise a venue. `ground_stats()` makes that
unnecessary for the baseline — see section 5 below.

Bulk fetching is still the right tool for questions Statsguru does not
answer, such as per-phase scoring across a whole tournament:

```python
import asyncio
from cricdata import AsyncCricinfoClient
from cricbot_views import fetch_many_scorecards

async def main():
    async with AsyncCricinfoClient() as aci:
        return await fetch_many_scorecards(aci, matches, concurrency=15)

pairs = asyncio.run(main())
```

Note that `AsyncSSR` does **not** cache, unlike the sync client. One
call per match is fine; fetching five views of the same match through
the async client would cost five requests, not one.


## 5. Venue baseline (Statsguru)

`pitch_profile()` above reads a single match. One match is noise — the
same figures expressed as a deviation from the venue's long-run baseline
are a signal.

`ground_stats()` returns venue aggregates in **one call**, which replaces
backfilling a season of scorecards.

Statsguru is HTML-scraped, slower and more brittle than the scorecard
endpoints, so every call goes through `safe_call()` — a venue lookup
failing must never break the rest of the notebook.


In [ ]:
from cricbot_stats import ground_id_from_match, ground_name_from_match, safe_call
from cricbot_venue import (
    venue_profile, format_venue_card,
    combine_match_and_history, format_pitch_read,
)

ground_id = ground_id_from_match(scorecard)
ground_name = ground_name_from_match(scorecard)
print('ground id:', ground_id, '|', ground_name)

# Statsguru wants ground.id, NOT ground.objectId.
raw = safe_call(ci.ground_stats, ground_id, fmt='t20')
venue = venue_profile(raw, name=ground_name)

print()
print(format_venue_card(venue))


### This match against the baseline

`deviation` is `None` when there is no baseline to compare against —
distinct from a deviation of zero. `confidence` reflects sample size:
below ~10 matches a venue figure says more about the teams than the pitch.


In [ ]:
pp = pitch_profile(innings)
combined = combine_match_and_history(pp, venue)

print(format_pitch_read(combined, pp))
print()
print(combined)


### Player record at this venue

`player_ground_stats()` splits a career by ground. Useful for previews:
"how does this batter go here" rather than in general.


In [ ]:
from cricbot_stats import resolve_player
from cricbot_venue import player_at_venue

found = resolve_player(ci, 'Virat Kohli')

if found:
    if found['ambiguous']:
        print('candidates:', found['candidates'])

    grounds = safe_call(
        ci.player_ground_stats, found['id'], fmt='t20', stat_type='batting'
    )
    at_venue = player_at_venue(grounds, found['display_name'], ground_name)

    for row in at_venue['rows'] or []:
        print(f"{row['ground']}: {row['matches']} inns, "
              f"avg {row['average']}, SR {row['strike_rate']}")
    if not at_venue['rows']:
        print('no rows for this venue')
else:
    print('player not found')


## 6. Rendered charts (optional)

The block-character charts are built for chat. When a real figure is
wanted, `scorecard_plots` renders the same data as PNGs.

Needs the optional extra: `pip install -r requirements-plots.txt`.

Note the win-probability x-axis is a **match-wide** sequence, not an
innings over number — it runs past 20 in a T20 because both innings are
concatenated. The innings break is marked.


In [ ]:
from scorecard_plots import render_match_report, PLOTS_AVAILABLE

if not PLOTS_AVAILABLE:
    print('matplotlib not installed — pip install -r requirements-plots.txt')
else:
    paths = render_match_report(scorecard, 'plots/', momentum_fn=momentum_series)
    for name, path in sorted(paths.items()):
        print(f'{name:26} {path}')


In [ ]:
# Display them inline
if PLOTS_AVAILABLE:
    from IPython.display import Image, display
    for name in ('win_probability', 'worm'):
        if name in paths:
            display(Image(filename=paths[name]))


## Open questions

1. `wicket_runs` calibration — see the sensitivity table above.
2. Momentum weights (`alpha`, `w_wkt`) are untuned. Validate against
   win-probability swings, which is the closest thing to ground truth.
3. Par SR is inflated in high-scoring games, so a good innings can score
   near zero. Consider par-by-phase instead of one match-wide number.
4. Entry bonus currently only applies above 15 runs — arbitrary.
5. No opposition-quality adjustment anywhere.
6. Venue baselines pool all formats unless `fmt` is set — always pass it.
7. No dew or toss adjustment yet; both confound venue scoring rates.
8. No opposition-strength adjustment, so a venue that hosts strong
   attacks reads as bowler-friendly.
